# Inverse Kinematics

Source orientation: printed pages 219-244, PDF pages 237-262. This notebook is an original visualization-first lesson inspired by the structure of *Modern Robotics: Mechanics, Planning, and Control* by Kevin M. Lynch and Frank C. Park. It does not quote or reproduce textbook prose, exercises, screenshots, or page crops.

## Chapter Question

How can we solve for joint coordinates when the forward map is nonlinear and sometimes many-to-one?

Forward kinematics is a map from joint space to task space. Inverse kinematics asks for a preimage: find a joint vector whose forward image has the desired end-effector pose. That sounds like algebra, but for most manipulators the equations are nonlinear, periodic, and branch dependent. A target can have several solutions, no exact solution, or a solution that is hard to approach from a given initial guess. The practical geometric question is therefore local: given the pose error we can see now, which tangent direction in joint space should we try next?

The notebook treats IK as root finding on a geometric residual. In a full spatial robot the residual is commonly represented as a body or space error twist. In the compact planar lab below, the residual is the end-effector position error, so the same idea is visible without six-dimensional notation. The Jacobian converts a small joint displacement into an approximate task displacement. Its pseudoinverse chooses a joint update, and damping limits that update when the Jacobian is nearly singular.

## Translation Guide

The chapter's ideas are translated into computational language using these terms: Newton-Raphson, body error twist, pseudoinverse, redundancy, closed loop, damping, convergence basin. The important conversions for this notebook are:

- Inverse kinematics is root finding on the forward kinematics residual.
- The Newton step linearizes the forward map at the current joint vector.
- The pseudoinverse chooses one local joint update among many possible updates.
- Redundant joints give a null-space direction that can change posture without changing the first-order task motion.
- Damping trades exact local correction for robustness near singularity.

The central habit is to keep the representation and the invariant together. A joint vector is a point on a configuration space, not just a list of numbers. A target pose is an element of task space. A residual is a vector that depends on the error convention. A Jacobian is a local derivative of the forward map, so it should explain a local change rather than magically solve the global nonlinear problem.

## Route Through the Notebook

1. Establish book-local paths and chapter metadata.
2. Build a dependency map for the chapter's vocabulary.
3. Generate an IK lab that compares several initial guesses for the same target.
4. Check a worked example: final residuals are small while the joint branches are different.
5. Sweep damping to see how it changes step size and convergence speed.
6. Finish with sanity checks that assert artifact existence, image variation, and numerical margins.

This is a standalone course notebook. The PDF can be useful for historical context and exercises, but the lesson here is complete without it. Definitions are restated in fresh language, examples are computed from scratch, and visuals are generated by course-local code. When the notebook mentions a source span, it is a navigation reference rather than a dependency for comprehension.

## Visualization Storyboard

| Concept | Representation | Artifact | Inspection target |
| --- | --- | --- | --- |
| concept dependency map | directed graph | `artifacts/chapter-06-inverse-kinematics/figures/concept-dependency-map.png` | which definitions feed the chapter's computation |
| damped planar inverse kinematics | endpoint paths and residual curves | `artifacts/chapter-06-inverse-kinematics/figures/inverse-kinematics-lab.png` | how an initial guess chooses a branch and how the residual shrinks |
| numeric invariant check | residual, branch, and damping margins | `artifacts/chapter-06-inverse-kinematics/figures/inverse-kinematics-checks.png` | small final errors, nonzero branch separation, and controlled singular updates |

The dependency map gives a quick view of how the chapter's terms fit together. The main lab makes IK visible: three initial guesses move toward the same target, and their residual curves reveal the local Newton behavior. One seed starts at a straight-arm singularity, where the first-order map loses a direction. The check figure records the numerical margins that should survive re-execution.

## Working Principles

A useful IK implementation should answer four questions. What residual is being minimized? Which frame or coordinate convention expresses it? What Jacobian linearizes that residual at the current configuration? What safeguard keeps the update meaningful near singularity or outside the convergence basin? The planar example below is intentionally small, but those four questions are the same ones used for a spatial manipulator with screw axes and homogeneous transforms.

Two geometric facts are worth watching in the plots. First, IK is local. A straight-line correction in task space does not imply a straight-line path in joint space, and a Newton update that helps near one posture may be a poor move somewhere else. Second, IK is branch dependent. For the same reachable target, different initial guesses can converge to visibly different postures. Those postures share the same endpoint but occupy different regions of configuration space.

## Applied Lab

Run a damped iterative IK update for a planar arm target and inspect the residual curve. Before changing the damping value, predict the tradeoff: weaker damping should make aggressive steps that can be fast but large near singularity; stronger damping should make smaller steps that are steadier but may require more iterations. The sweep near the end of the notebook turns that prediction into numbers.

## Pitfalls To Watch

- A local method can converge to the wrong branch or fail outside its basin.
- Small task error can require large joint motion near singularity.
- A residual must match the Jacobian convention; mixing body, space, and coordinate errors can look plausible while moving the wrong way.
- Damping is not a proof of convergence. It is a local regularizer, so the target and initial guess still matter.

## Takeaways

- IK is numerical geometry on a nonlinear forward map.
- Rank, damping, and initial guess decide much of the behavior.
- A small residual in task space does not identify a unique joint configuration.
- Good IK code reports margins, not just a final joint vector.

By the end of this notebook, the reader should be able to explain why inverse kinematics is solved iteratively, interpret a damped pseudoinverse update, recognize branch dependence, and state numerical checks that would catch a broken implementation.


In [ ]:
from pathlib import Path
import json
import math
import sys

import numpy as np

HERE = Path.cwd().resolve()
BOOK_ROOT = next(
    parent for parent in [HERE, *HERE.parents]
    if (parent / "AGENTS.md").exists() and (parent / "Mordern Robotics.pdf").exists()
)
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import display_artifact, save_json
from utils.validation import assert_artifact, image_stats

print(f"BOOK_ROOT={BOOK_ROOT}")

In [ ]:
CHAPTER = {
  "number": 6,
  "slug": "chapter-06-inverse-kinematics",
  "title": "Inverse Kinematics",
  "notebook": "06-inverse-kinematics.ipynb",
  "printed_start": 219,
  "printed_end": 244,
  "pdf_start": 237,
  "pdf_end": 262,
  "part_slug": "part-02-manipulator-kinematics",
  "part_title": "Manipulator Kinematics",
  "theme": "kinematics",
  "visual_focus": "damped planar inverse kinematics",
  "visual_kind": "endpoint paths and residual curves",
  "artifact_stem": "inverse-kinematics",
  "inspection_target": "how an initial guess chooses a branch and how the residual shrinks",
  "question": "How can we solve for joint coordinates when the forward map is nonlinear and sometimes many-to-one?",
  "terms": [
    "Newton-Raphson",
    "body error twist",
    "pseudoinverse",
    "redundancy",
    "closed loop",
    "damping",
    "convergence basin"
  ],
  "translation": [
    "Inverse kinematics is root finding on the forward kinematics residual.",
    "The Newton step linearizes the forward map at the current joint vector.",
    "The pseudoinverse chooses one local joint update among many possibilities.",
    "Damping trades exact correction for robustness near singularity."
  ],
  "lab": "Run a damped iterative IK update for a planar arm target and inspect the residual curve.",
  "pitfalls": [
    "A local method can converge to the wrong branch or fail outside its basin.",
    "Small task error can require large joint motion near singularity.",
    "A residual must match the Jacobian convention used to build the update."
  ],
  "takeaways": [
    "IK is numerical geometry on a nonlinear map.",
    "Rank, damping, and initial guess decide much of the behavior.",
    "A small task residual does not identify a unique joint configuration."
  ]
}

from utils.visuals import build_storyboard
storyboard = build_storyboard(CHAPTER)
storyboard


In [ ]:
from utils.visuals import build_chapter_visuals
from utils.artifacts import save_matplotlib
from utils.kinematics import planar_arm_points, planar_jacobian, manipulability_measure

import matplotlib.pyplot as plt

lengths = np.array([1.0, 0.75, 0.45])
target = np.array([1.35, 0.65])
start_configs = {
    "straight singular seed": np.array([0.0, 0.0, 0.0]),
    "elbow-up seed": np.array([0.1, 0.4, -0.6]),
    "elbow-down seed": np.array([1.7, -1.6, 0.8]),
}


def wrap_to_pi(theta):
    return (theta + np.pi) % (2 * np.pi) - np.pi


def damped_least_squares(J, damping):
    if damping == 0:
        return np.linalg.pinv(J)
    regularized = J @ J.T + damping**2 * np.eye(J.shape[0])
    return J.T @ np.linalg.solve(regularized, np.eye(J.shape[0]))


def run_damped_ik(start, damping=0.08, max_steps=45, tolerance=1e-4, max_step=0.45):
    theta = np.array(start, dtype=float)
    history = []
    for step_index in range(max_steps + 1):
        points = planar_arm_points(lengths, theta)
        J = planar_jacobian(lengths, theta)
        error = target - points[-1]
        residual = float(np.linalg.norm(error))
        history.append({
            "theta": theta.copy(),
            "endpoint": points[-1].copy(),
            "residual": residual,
            "rank": int(np.linalg.matrix_rank(J)),
            "manipulability": float(manipulability_measure(J)),
            "step_norm": 0.0,
        })
        if residual < tolerance:
            break
        update = damped_least_squares(J, damping) @ error
        step_norm = float(np.linalg.norm(update))
        if step_norm > max_step:
            update = update * (max_step / step_norm)
        history[-1]["step_norm"] = step_norm
        theta = wrap_to_pi(theta + update)
    return history


outputs = build_chapter_visuals(CHAPTER)
ik_runs = {name: run_damped_ik(start) for name, start in start_configs.items()}
colors = {
    "straight singular seed": "#d95f02",
    "elbow-up seed": "#1b9e77",
    "elbow-down seed": "#7570b3",
}

fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.9), facecolor="#f8f5ef")
ax = axes[0]
outer = plt.Circle((0.0, 0.0), lengths.sum(), fill=False, ls="--", lw=1.2, color="#8a8f98", label="reach boundary")
inner_radius = max(lengths.max() - (lengths.sum() - lengths.max()), 0.0)
inner = plt.Circle((0.0, 0.0), inner_radius, fill=False, ls=":", lw=1.2, color="#8a8f98", label="inner boundary")
ax.add_patch(outer)
ax.add_patch(inner)
ax.scatter([target[0]], [target[1]], s=95, marker="*", color="#c0392b", label="target")
for name, history in ik_runs.items():
    endpoint_path = np.array([item["endpoint"] for item in history])
    final_theta = history[-1]["theta"]
    final_points = planar_arm_points(lengths, final_theta)
    initial_points = planar_arm_points(lengths, start_configs[name])
    ax.plot(endpoint_path[:, 0], endpoint_path[:, 1], "o-", ms=3.5, lw=1.8, color=colors[name], label=name)
    ax.plot(initial_points[:, 0], initial_points[:, 1], "--", lw=1.0, color=colors[name], alpha=0.35)
    ax.plot(final_points[:, 0], final_points[:, 1], "-", lw=2.6, color=colors[name], alpha=0.9)
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(-0.35, 2.35)
ax.set_ylim(-0.35, 1.65)
ax.set_title("same target, different initial guesses")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.grid(alpha=0.25)
ax.legend(loc="lower right", fontsize=8)

ax = axes[1]
for name, history in ik_runs.items():
    residuals = [item["residual"] for item in history]
    ax.semilogy(range(len(residuals)), residuals, "o-", lw=2.0, ms=4, color=colors[name], label=name)
ax.axhline(1e-4, color="#4d4d4d", lw=1.0, ls="--", label="tolerance")
ax.set_title("residual contraction")
ax.set_xlabel("iteration")
ax.set_ylabel("endpoint error norm")
ax.grid(alpha=0.25, which="both")
ax.legend(fontsize=8)
fig.suptitle("Damped inverse kinematics for a redundant planar arm", y=1.02)
fig.tight_layout()
ik_lab_path = save_matplotlib(fig, CHAPTER["slug"], "figures", "inverse-kinematics-lab.png")
plt.close(fig)

final_residuals = {name: history[-1]["residual"] for name, history in ik_runs.items()}
min_manipulability = {name: min(item["manipulability"] for item in history) for name, history in ik_runs.items()}
final_thetas = {name: history[-1]["theta"] for name, history in ik_runs.items()}
branch_separation = float(np.linalg.norm(wrap_to_pi(final_thetas["elbow-up seed"] - final_thetas["elbow-down seed"])))
undamped_probe = run_damped_ik(start_configs["straight singular seed"], damping=0.0, max_steps=80)
damped_probe = ik_runs["straight singular seed"]
undamped_peak_update = float(max(item["step_norm"] for item in undamped_probe))
damped_peak_update = float(max(item["step_norm"] for item in damped_probe))
outputs["figures"][1] = ik_lab_path
outputs["metrics"] = {
    "worst_final_residual": float(max(final_residuals.values())),
    "branch_joint_separation": branch_separation,
    "straight_seed_min_manipulability": float(min_manipulability["straight singular seed"]),
    "undamped_peak_update_norm": undamped_peak_update,
    "damped_peak_update_norm": damped_peak_update,
}

fig, ax = plt.subplots(figsize=(8.6, 4.2), facecolor="#f8f5ef")
check_values = {
    "worst final residual": outputs["metrics"]["worst_final_residual"],
    "branch separation": outputs["metrics"]["branch_joint_separation"],
    "damped peak update": outputs["metrics"]["damped_peak_update_norm"],
    "undamped peak update": outputs["metrics"]["undamped_peak_update_norm"],
}
bar_colors = ["#1b9e77", "#7570b3", "#d95f02", "#e6ab02"]
ax.barh(list(check_values.keys()), list(check_values.values()), color=bar_colors)
ax.set_xscale("log")
ax.set_title("IK numerical margins")
ax.set_xlabel("log-scaled magnitude")
ax.grid(alpha=0.25, axis="x", which="both")
fig.tight_layout()
ik_check_path = save_matplotlib(fig, CHAPTER["slug"], "figures", "inverse-kinematics-checks.png")
plt.close(fig)
outputs["figures"][2] = ik_check_path
outputs["checks"] = save_json(
    {
        "metrics": outputs["metrics"],
        "final_residuals": final_residuals,
        "image_stats": {path.name: image_stats(path) for path in outputs["figures"]},
    },
    CHAPTER["slug"],
    "checks",
    "final-sanity.json",
)

for artifact in outputs["figures"]:
    display_artifact(artifact, width=760)
outputs["metrics"]


## Worked Example

The worked example checks two IK facts that are easy to miss when only the final joint vector is printed. First, all three runs reach the target to numerical tolerance. Second, the elbow-up and elbow-down seeds end at different joint configurations, even though their endpoints agree. The target-space residual is small, but the inverse image is not unique.


In [ ]:
branch_errors = {
    name: float(np.linalg.norm(planar_arm_points(lengths, history[-1]["theta"])[-1] - target))
    for name, history in ik_runs.items()
}
branch_gap = float(np.linalg.norm(wrap_to_pi(final_thetas["elbow-up seed"] - final_thetas["elbow-down seed"])))
first_residual_drop = {
    name: float(history[0]["residual"] - history[min(2, len(history) - 1)]["residual"])
    for name, history in ik_runs.items()
}
worked_example = {
    "target": target.round(4).tolist(),
    "final_residuals": branch_errors,
    "branch_joint_separation": branch_gap,
    "straight_seed_initial_rank": ik_runs["straight singular seed"][0]["rank"],
    "straight_seed_initial_manipulability": ik_runs["straight singular seed"][0]["manipulability"],
    "first_two_step_residual_drop": first_residual_drop,
}
assert max(branch_errors.values()) < 1e-3
assert branch_gap > 1.0
worked_example


## Applied Lab

Now vary the damping parameter while keeping the singular straight-arm seed fixed. The prediction is qualitative: stronger damping should reduce the largest raw joint update near the singularity, while usually taking more iterations to reach the same tolerance. The final residual should remain small for this reachable target.


In [ ]:
damping_values = [0.0, 0.02, 0.08, 0.2, 0.4, 0.7]
damping_sweep = {}
for damping in damping_values:
    history = run_damped_ik(start_configs["straight singular seed"], damping=damping, max_steps=80)
    damping_sweep[f"damping={damping:g}"] = {
        "iterations": len(history) - 1,
        "final_residual": float(history[-1]["residual"]),
        "max_raw_step_norm": float(max(item["step_norm"] for item in history)),
        "final_theta": history[-1]["theta"].round(4).tolist(),
    }

lab_summary = {
    "theme": CHAPTER["theme"],
    "target": target.round(4).tolist(),
    "damping_sweep": damping_sweep,
}
assert all(item["final_residual"] < 1e-3 for item in damping_sweep.values())
lab_summary


## Sanity Checks

The final cell asserts that the generated artifacts exist, are large enough to be useful, and have nontrivial pixel variation. It also stores a JSON summary under the chapter's artifact subtree so the notebook leaves a machine-checkable trace.

In [ ]:
artifact_stats = {}
for artifact in outputs["figures"]:
    resolved = assert_artifact(artifact)
    stats = image_stats(resolved)
    assert stats["pixel_std"] > 2.0, f"low variation image: {resolved}"
    artifact_stats[resolved.name] = stats
assert_artifact(outputs["checks"], min_size=100)
assert outputs["metrics"]["worst_final_residual"] < 1e-3
assert outputs["metrics"]["branch_joint_separation"] > 1.0
assert outputs["metrics"]["undamped_peak_update_norm"] > outputs["metrics"]["damped_peak_update_norm"]
sanity = {
    "chapter": CHAPTER["slug"],
    "metrics": outputs["metrics"],
    "worked_example": worked_example,
    "lab_summary": lab_summary,
    "artifact_stats": artifact_stats,
}
sanity_path = save_json(sanity, CHAPTER["slug"], "checks", "notebook-sanity.json")
display_artifact(sanity_path)
sanity
